In [ ]:
!pip install ucimlrepo

In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

data = fetch_ucirepo(id=327)
X = data.data.features
y = data.data.targets

print(X.shape)
print(y.head())

(11055, 30)
   result
0      -1
1      -1
2      -1
3      -1
4       1


In [ ]:
print(X.head())

   having_ip_address  url_length  shortining_service  having_at_symbol  \
0                 -1           1                   1                 1   
1                  1           1                   1                 1   
2                  1           0                   1                 1   
3                  1           0                   1                 1   
4                  1           0                  -1                 1   

   double_slash_redirecting  prefix_suffix  having_sub_domain  sslfinal_state  \
0                        -1             -1                 -1              -1   
1                         1             -1                  0               1   
2                         1             -1                 -1              -1   
3                         1             -1                 -1              -1   
4                         1             -1                  1               1   

   domain_registration_length  favicon  ...  rightclick  popupwindow

In [ ]:
print(y.value_counts())

result
 1        6157
-1        4898
Name: count, dtype: int64


In [ ]:
print("Missing values:", X.isnull().sum().sum())
print("Duplicate rows:", X.duplicated().sum())

Missing values: 0
Duplicate rows: 5270


In [ ]:
df = X.copy()
df["label"] = (y.iloc[:, 0] == -1).astype(int)

df = df.drop_duplicates()

print("Table size after cleaning:", df.shape)
print(df["label"].value_counts())

Table size after cleaning: (5849, 31)
label
1    3019
0    2830
Name: count, dtype: int64


In [ ]:
from sklearn.model_selection import train_test_split

features = df.drop(columns="label")
labels = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    features, labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

print("Training rows:", X_train.shape[0])
print("Test rows:", X_test.shape[0])

Training rows: 4679
Test rows: 1170


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# Model 1: Logistic Regression
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
print("=== Logistic Regression ===")
print(classification_report(y_test, lr.predict(X_test), target_names=["safe", "phishing"]))

# Model 2: Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
print("=== Random Forest ===")
print(classification_report(y_test, rf.predict(X_test), target_names=["safe", "phishing"]))

=== Logistic Regression ===
              precision    recall  f1-score   support

        safe       0.91      0.92      0.91       566
    phishing       0.92      0.91      0.92       604

    accuracy                           0.92      1170
   macro avg       0.92      0.92      0.92      1170
weighted avg       0.92      0.92      0.92      1170

=== Random Forest ===
              precision    recall  f1-score   support

        safe       0.92      0.95      0.94       566
    phishing       0.96      0.93      0.94       604

    accuracy                           0.94      1170
   macro avg       0.94      0.94      0.94      1170
weighted avg       0.94      0.94      0.94      1170



In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

In [ ]:
from tensorflow import keras

model = keras.Sequential([
    keras.layers.Input(shape=(X_train_s.shape[1],)),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(32, activation="relu"),
    keras.layers.Dense(1, activation="sigmoid"),
])

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True
)

history = model.fit(
    X_train_s, y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=32,
    callbacks=[early_stop],
)

Epoch 1/50
117/117 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.8178 - loss: 0.4233 - val_accuracy: 0.9071 - val_loss: 0.2449
Epoch 2/50
117/117 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.9065 - loss: 0.2316 - val_accuracy: 0.9263 - val_loss: 0.2002
Epoch 3/50
117/117 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9207 - loss: 0.1973 - val_accuracy: 0.9284 - val_loss: 0.1913
Epoch 4/50
117/117 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9257 - loss: 0.1875 - val_accuracy: 0.9306 - val_loss: 0.1838
Epoch 5/50
117/117 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9249 - loss: 0.1754 - val_accuracy: 0.9263 - val_loss: 0.1830
Epoch 6/50
117/117 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9265 - loss: 0.1704 - val_accuracy: 0.9284 - val_loss: 0.1780
Epoch 7/50
117/117 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9303 - loss: 0.1642 - val_accuracy: 0.9316 - val_loss: 0.1709
Epoch 8/50
117/117 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9329 - loss: 0.1597 - val_accuracy: 

In [ ]:
from sklearn.metrics import classification_report

pred = (model.predict(X_test_s) > 0.5).astype(int).ravel()
print(classification_report(y_test, pred, target_names=["safe", "phishing"]))

37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
              precision    recall  f1-score   support

        safe       0.93      0.94      0.93       566
    phishing       0.94      0.94      0.94       604

    accuracy                           0.94      1170
   macro avg       0.94      0.94      0.94      1170
weighted avg       0.94      0.94      0.94      1170



In [ ]:
import joblib

model.save("phishing_model.keras")
joblib.dump(scaler, "scaler.joblib")

# also save the column names, in order
joblib.dump(list(X_train.columns), "feature_names.joblib")
print("Saved!")

Saved!
